# Homework 3 — Star Schema per le Financial Transactions

**Course:** Big Data Analytics  
**Professor:** Giovanni Morana  
**Student:** Carmen Padova  
**Student ID:** 1000035594

---------------------------------------------------------------------------------------------------------------------------------------

**Datasets**: 
- `account-statement-1-1-2024-12-31-2024.csv`: l'estratto conto di un conto titoli per il 2024 
- `symbols.csv`: l'anagrafica dei titoli 
- `country.csv`: la classificazione geografica dei paesi 
---------------------------------------------------------------------------------------------------------------------------------------

# Part 1 — Data Warehouse Modeling: Star Schema

## 1.1 Define the Business Process and the Fact Grain

**The business process.** The business process under analysis is the **trading of financial instruments** on a brokerage account during 2024, which is also the historical interval covered by the data mart; the account records BUY, SELL and DIVIDEND operations.

There is a single process, and all three sources relate to it, but with different roles:

- `account-statement-1-1-2024-12-31-2024.csv` records its events;
- `symbols.csv` and `country.csv` describe structural properties of the domain, which will serve as the analysis axes.

**The fact grain.** Before declaring the grain, the fact itself must be identified. In the DFM a fact is the concept that gathers the category of events we want to analyze, and a good fact must have two characteristics:

1. it happens dynamically over time (it is not a static property of the domain);
2. it is relevant to the decision-making process.

This criterion justifies the roles assigned above to the three sources, by looking at their nature:

- `account-statement-1-1-2024-12-31-2024.csv` records things that *happen*: each row is an operation that took place at a specific moment, and the archive grows as the account is traded. It is the only dynamic archive, so it is the natural candidate to express the **fact**;
- `symbols.csv` and `country.csv` do not record events but stable *descriptions* of the domain (which company, industry and sector a ticker belongs to; how a country is classified geographically). They are nearly-static archives: candidates for **dimensions**, not facts.

Once the fact is defined, the grain answers the question: what does a single row of the fact table represent? Here it is fixed by the assignment:

> *one row in `Fact_Transactions` = one single transaction from `account-statement-1-1-2024-12-31-2024.csv`*

Each fact row therefore corresponds to a single business event, at the same granularity as the operational source. A fact table of this kind is called an **event fact** (or transaction fact). It is the finest possible grain, the one that gives maximum flexibility for analysis; with a few thousand rows per year there are none of the performance issues that in other contexts would push towards a more aggregated grain.

## 1.2 Identify Fact and Dimensions

The analytical questions listed in section 2.2 act as the requirement analysis to decide which source attributes are relevant.

In the DFM a **measure** is a numerical property of the fact that describes a quantitative aspect relevant to the analysis, while a **<u>dimension</u>** is a property with a finite domain that describes an analysis axis of the fact; attributes that are neither can remain in the model as ***descriptive attributes***. Applying these criteria to the source attributes:

| Source attribute | Datasets | Destination | Role | Notes |
|---|---|---|---|---|
| `Unit` | account statement | `Fact_Transactions` | **measure** | flow measure (quantity traded in the event), therefore additive along all dimensions with the SUM operator |
| `IDTransaction` | account statement | `Fact_Transactions` | *descriptive attribute* of the fact | it is neither a measure nor a dimension, and in the data it does not uniquely identify a transaction (same ID on several operations) |
| `Date` | account statement | `Dim_Time` | temporal <u>dimension</u> | time is a mandatory dimension in a data mart ("Time should always be a dimension"): it enables trends and comparisons across periods, and it is the root of the time hierarchy |
| `TransactionType` | account statement | `Dim_TransactionType` | <u>dimension</u> | finite domain {BUY, SELL, DIVIDEND}: a typical analysis axis for selecting and grouping by operation type |
| `Symbol` | account statement / symbols | `Dim_Symbol` | <u>dimension</u> | it is the reference the fact uses to identify the security, and it is unique in the symbols dataset |
| `company_name` | symbols | `Dim_Symbol` | *descriptive attribute* of symbol | 1:1 with the symbol: it does not define an additional aggregation level, it only makes results readable |
| `industry`, `sector` | symbols | `Dim_Symbol` | dimensional attributes | upper levels of the symbol hierarchy |
| `country` (symbols) = `name` (country) | symbols / country | `Dim_Geography` | <u>dimension</u> | same concept in two datasets: placed only in `Dim_Geography` to avoid duplication across dimensions |
| `sub-region`, `region` | country | `Dim_Geography` | dimensional attributes | upper levels of the geographic hierarchy |
| `alpha-2`, `alpha-3`, `country-code`, `iso_3166-2`, `intermediate-region`, `region-code`, `sub-region-code`, `intermediate-region-code` | country | — | discarded | not required by any question in 2.2 (pruning) |

The fact is therefore surrounded by four <u>dimensions</u> — **Time**, **Geography**, **Symbol**, **TransactionType**.

## 1.3 Define Dimension Hierarchies

Hierarchies are subtrees rooted in the dimensions: the nodes are dimensional attributes with discrete values, and the arcs represent **functional dependencies** between pairs of attributes. Moving along a hierarchy from its root yields progressively more aggregated granularities, and this is what makes roll-up and drill-down operations possible at OLAP time.

![Fact schema TRANSACTION](images/fact_schema.png)

**Dim_Time** — the hierarchy required by the assignment: each day belongs to exactly one month, each month to one quarter, each quarter to one year.

`DayOfWeek` is not on the main path of the hierarchy: it is a secondary branch starting from date (like the `day`/`week`/`holiday` branches in the SALE example from the slides), used for selections such as "Mondays only", not for rolling up towards month or quarter.

**Dim_Geography** — here the assignment contains an inconsistency: it lists the hierarchy as *Country → Region → Sub-region*. In the `country.csv` dataset, however, each sub-region is a subset of a region (for instance *Southern Europe* belongs to *Europe*): the functional dependencies are `country → sub-region` and `sub-region → region`, so the sub-region level is **finer** than the region level. Since hierarchy levels must go from the finest to the most aggregated, the correct order is **Country → Sub-region → Region**.

**Dim_Symbol** — the symbols dataset reveals a natural hierarchy: each symbol belongs to exactly one industry and each industry to exactly one sector, so the functional dependencies support the three-level hierarchy. `CompanyName` is 1:1 with the symbol and is not used for aggregation: it is a descriptive attribute.

**Dim_TransactionType** — the domain is {BUY, SELL, DIVIDEND} and no attribute groups these values: the dimension has a single level and no hierarchy. This is an expected case: the dimension is still used for selection and grouping, it simply offers no roll-up levels.

## 1.4 Design the Star Schema

![Star Schema](images/star_schema.png)

**1) Surrogate keys for the dimensions.** Each dimension table has a surrogate key as primary key, generated as a progressive integer during the ETL phase:

| Dimension table | Surrogate key (PK) |
|---|---|
| `Dim_Time` | `Time_SK` |
| `Dim_Geography` | `Geography_SK` |
| `Dim_Symbol` | `Symbol_SK` |
| `Dim_TransactionType` | `TransactionType_SK` |

**2) Fact table foreign keys.** `Fact_Transactions` carries one foreign key per dimension, each referencing the corresponding surrogate key. The mapping from the source data is:

| Foreign key | References | Valued from |
|---|---|---|
| `Time_SK` | `Dim_Time` | the `Date` of the transaction (day granularity) |
| `Symbol_SK` | `Dim_Symbol` | the `Symbol` of the transaction |
| `Geography_SK` | `Dim_Geography` | the country of the issuing company, obtained through the symbol |
| `TransactionType_SK` | `Dim_TransactionType` | the `TransactionType` of the transaction |

The primary key of the fact table deserves a remark: if we had used the composition of the foreign keys as PK, then — since at the grain of the single transaction two distinct transactions can coincide on all four dimensions (same day, same symbol, same type) — the composition would not be unique. The fact table therefore gets its own surrogate key `Transaction_SK`; `IDTransaction` cannot play this role because, as observed in 1.2, it contains duplicates in the source file, so it would not be unique either.

**3) Measures and descriptive attributes.** `Unit` is the only measure. The descriptive attributes are `IDTransaction` on the fact table and `CompanyName` in `Dim_Symbol`; they carry information but are not used for aggregation.

The resulting tables:

| Table | Primary key | Foreign keys | Other attributes |
|---|---|---|---|
| `Fact_Transactions` | `Transaction_SK` (surrogate) | `Time_SK`, `Geography_SK`, `Symbol_SK`, `TransactionType_SK` | `IDTransaction` (descriptive), `Unit` (**measure**) |
| `Dim_Time` | `Time_SK` (surrogate) | — | `Date`, `DayOfWeek`, `Month`, `Quarter`, `Year` |
| `Dim_Geography` | `Geography_SK` (surrogate) | — | `Country`, `SubRegion`, `Region` |
| `Dim_Symbol` | `Symbol_SK` (surrogate) | — | `Symbol`, `CompanyName` (descriptive), `Industry`, `Sector` |
| `Dim_TransactionType` | `TransactionType_SK` (surrogate) | — | `TransactionType` |

# Part 2 — Data Transformation and Analysis

## 2.1 Load and Clean the Data

In [2]:
# ============================================================
# 1. Import libraries
# ============================================================

# pandas is the only library we need: it covers loading, cleaning,
# joining and aggregating the data.
import pandas as pd

In [3]:
# ============================================================
# 2. Load the three source datasets
# ============================================================

# The three files do not share the same format, so different read options are needed:
# - <account statement> and <symbols.csv> separate columns with ";" instead of
#   the comma: without sep=";" pandas would find no separators and would load
#   each line as one single column;
#   moreover, these two files start with an invisible character that some
#   programs (e.g. Excel) add at the top of the file when saving it:
#   if not removed, it sticks to the name of the first column
#   ("\ufeffIDTransaction" instead of "IDTransaction") and column
#   references fail. encoding="utf-8-sig" removes it while reading;
# - <country.csv> use the standard "," separated CSV, so I can use the default options.

transactions = pd.read_csv('Datasets/account-statement-1-1-2024-12-31-2024.csv',
                           sep=';', encoding='utf-8-sig')
symbols = pd.read_csv('Datasets/symbols.csv', sep=';', encoding='utf-8-sig')
country = pd.read_csv('Datasets/country.csv')

# Show the shape of each dataframe: rows and columns.
print('transactions:', transactions.shape)
print('symbols:     ', symbols.shape)
print('country:     ', country.shape)

# Preview the first rows of the fact source.
transactions.head()

transactions: (2745, 6)
symbols:      (3194, 5)
country:      (249, 11)


,IDTransaction,Date,TransactionType,Symbol,Unit,Unnamed: 5
0,2.769834e+09,11/01/2024 10:44:03,BUY,BAP,1605.0,NaN
1,2.767325e+09,24/01/2024 08:07:24,SELL,BAP,1605.0,NaN
2,2.815474e+09,10/01/2024 11:00:08,SELL,BAP,914.0,NaN
3,2.622244e+09,16/01/2024 08:14:21,BUY,ACGL,646.0,NaN
4,2.629871e+09,16/01/2024 14:34:12,SELL,ALVO,646.0,NaN


### Check 1 — Missing values

Before transforming anything, we must understand the quality of the sources so I look for missing values in all three dataframes.

In [4]:
# ============================================================
# 3. Check missing values in all three sources
# ============================================================

# Missing values per column in the account statement.
print('--- transactions ---')
print(transactions.isna().sum())

# Count the rows where EVERY column is NaN (carry no event).
print('Fully empty rows:', transactions.isna().all(axis=1).sum())

# Total missing values in the symbols master data.
print('\n--- symbols ---')
print('Total missing values:', symbols.isna().sum().sum())

# Missing values in the columns of country.csv used by the model.
print('\n--- country ---')
print(country[['name', 'sub-region', 'region']].isna().sum())

# Show WHICH countries have a missing region.
print('Rows with missing region:', country.loc[country['region'].isna(), 'name'].tolist())

--- transactions ---
IDTransaction       464
Date                464
TransactionType     464
Symbol              464
Unit                464
Unnamed: 5         2745
dtype: int64
Fully empty rows: 464

--- symbols ---
Total missing values: 0

--- country ---
name          0
sub-region    2
region        2
dtype: int64
Rows with missing region: ['Antarctica', 'Taiwan, Province of China']


The check highlights three things:

- **`transactions`**: 464 fully empty rows (no value in any column: they represent no transaction and must be removed) and a fully empty column `Unnamed: 5`, created by pandas because of the trailing `;` on every line of the file (it will be removed in check 2 together with the other unused attributes);
- **`symbols`**: no missing values;
- **`country`**: two countries have empty `region` and `sub-region` cells (*Antarctica* and *Taiwan, Province of China*). Antarctica is irrelevant here, since no company in the portfolio is based there. Taiwan isn't because account statement contains securities of Taiwanese companies, so that defective row would enter the star schema. If the region were left empty, any `groupby` on region or sub-region would silently drop those transactions (pandas excludes NaN groups) and every geographic analysis would be distorted. I'll fix it in check 4 which is dedicated.

In [4]:
# ============================================================
# 4. Remove the fully empty rows
# ============================================================

# Drop the rows where all the columns are NaN and rebuild the index.
transactions = transactions.dropna(how='all').reset_index(drop=True)

# Verify the result: row count and residual missing values
# (excluding the spurious column, which is removed later).
print('Remaining rows:', len(transactions))
print('Residual missing values (excl. Unnamed: 5):',
      transactions.drop(columns=['Unnamed: 5']).isna().sum().sum())

Remaining rows: 2281
Residual missing values (excl. Unnamed: 5): 0


In [5]:
# ============================================================
# 5. Inspect the domains of the model attributes
# ============================================================

# Distribution of the transaction types: we expect BUY / SELL / DIVIDEND.
print(transactions['TransactionType'].value_counts())

# Is IDTransaction unique? If not, it cannot be a primary key.
print('\nDuplicated IDTransaction values:',
      transactions['IDTransaction'].duplicated().sum())

TransactionType
SELL        1089
BUY         1082
DIVIDENT     110
Name: count, dtype: int64

Duplicated IDTransaction values: 1145


Two value corrections are needed:

- `DIVIDENT` is a typo for `DIVIDEND`, so we fix the label. Dividend transactions **stay** in the fact table, because the grain says "one row per transaction of the account statement";
- `Unit` and `IDTransaction` were read as floats but contain only integers, so we cast them back.

Moreover, `IDTransaction` shows 1145 duplicates on clearly different transactions (different dates and symbols) but the rows are **not** removed, because they are distinct events, not duplicated records. This is the empirical confirmation that the attribute cannot act as a key, as already staied.

In [6]:
# ============================================================
# 6. Fix the value errors found above
# ============================================================

# Fix the typo in the transaction type label.
transactions['TransactionType'] = transactions['TransactionType'].replace('DIVIDENT', 'DIVIDEND')

# Cast the numeric columns back to integers.
transactions['Unit'] = transactions['Unit'].astype(int)
transactions['IDTransaction'] = transactions['IDTransaction'].astype(int)

# Verify: the three types are now BUY / SELL / DIVIDEND.
print(transactions['TransactionType'].value_counts())

TransactionType
SELL        1089
BUY         1082
DIVIDEND     110
Name: count, dtype: int64


### Check 2 — Remove unused attributes

We keep in each dataframe only the attributes included in the dimensional model, so `Unnamed: 5` from the account statement, the ISO codes and technical columns from `country.csv` are excluded.

In [5]:
# ============================================================
# 7. Keep only the attributes of the dimensional model
# ============================================================

# Account statement: only the five attributes mapped in 1.2.
transactions = transactions[['IDTransaction', 'Date', 'TransactionType', 'Symbol', 'Unit']]

# Symbols master data: ticker, company, industry, sector and country
# (country is kept here only as a bridge towards Dim_Geography).
symbols = symbols[['symbol', 'company_name', 'industry', 'sector', 'country']]

# Country classification: name plus the two hierarchy levels.
country = country[['name', 'sub-region', 'region']]

# Verify the remaining columns.
print('transactions:', transactions.columns.tolist())
print('symbols:     ', symbols.columns.tolist())
print('country:     ', country.columns.tolist())

transactions: ['IDTransaction', 'Date', 'TransactionType', 'Symbol', 'Unit']
symbols:      ['symbol', 'company_name', 'industry', 'sector', 'country']
country:      ['name', 'sub-region', 'region']


### Check 3 — Every transaction symbol exists in the symbols dataset

Every fact row must have its four foreign keys valued, and two of them depend on the symbols master data: `Symbol_SK` is obtained by looking up the ticker in `Dim_Symbol`, while `Geography_SK` is obtained by going from the ticker to the company country, which also lives in `symbols.csv`. If a ticker is missing from the master data the effect is double: there is no `Dim_Symbol` row to point to, and the company country is unknown as well, so two FKs out of four would stay empty. Such a row violates referential integrity and would disappear from every analysis by sector, industry or geography anyway, because the join with the dimensions would find no match.

In [8]:
# ============================================================
# 8. Referential check: transaction symbols vs symbols dataset
# ============================================================

# Set difference: tickers traded in 2024 that do NOT exist in the master data.
unknown = set(transactions['Symbol']) - set(symbols['symbol'])

# How many transaction rows are affected?
n_bad = transactions['Symbol'].isin(unknown).sum()

print(f'Symbols not found in the master data: {len(unknown)}')
print(sorted(unknown))
print(f'Affected rows: {n_bad} out of {len(transactions)}')

Symbols not found in the master data: 18
['AGO.l', 'ARCH', 'AZM', 'CCAP', 'CSIQ', 'FNC', 'HTGC', 'IBE', 'MFG', 'MONC', 'OBDC', 'RCMT', 'RIGZU', 'SAP', 'TKC', 'UCG', 'VWS', 'WF']
Affected rows: 212 out of 2281


18 symbols (212 transactions, about 9%) do not exist in the master data: they are erroneous records with respect to the model. Between the two error handling strategies seen in class ("stop transformation" or "process valid data only") we adopt the second one: the unmappable rows are excluded, documenting their number.

In [9]:
# ============================================================
# 9. Process valid data only
# ============================================================

# Keep only the transactions whose ticker exists in the master data.
transactions = transactions[transactions['Symbol'].isin(symbols['symbol'])].reset_index(drop=True)

print('Valid rows loadable into the fact table:', len(transactions))

Valid rows loadable into the fact table: 2069


### Check 4 — Every company country can be mapped to the country dataset

In [10]:
# ============================================================
# 10. Referential check: company countries vs country dataset
# ============================================================

# Set difference: country names used in symbols.csv that do not
# match any name in country.csv.
print('Unmappable countries:', sorted(set(symbols['country']) - set(country['name'])))

Unmappable countries: ['Taiwan', 'Turkey']


`Taiwan` and `Turkey` do not match because `country.csv` uses the ISO names ***Taiwan, Province of China*** and ***Türkiye***: the same concept represented differently in two sources, a conflict solved with a value translation during integration.

Moreover, as found in check 1, the Taiwan row of `country.csv` has an empty `region` and `sub-region`: leaving them null would silently drop the Taiwanese transactions from every analysis by region or sub-region. We apply the correct values (`Asia` / `Eastern Asia`, the classification used by the other East Asian countries), as done in the cleansing phase when default values are applied to missing fields.

In [11]:
# ============================================================
# 11. Fix the country values
# ============================================================

# Translate the two country names to the ISO naming used by country.csv.
symbols['country'] = symbols['country'].replace({
    'Taiwan': 'Taiwan, Province of China',
    'Turkey': 'Türkiye'
})

# Fill the missing region / sub-region of Taiwan with the correct values.
country.loc[country['name'] == 'Taiwan, Province of China',
            ['sub-region', 'region']] = ['Eastern Asia', 'Asia']

# Verify: no unmappable country is left.
print('Unmappable countries after the fix:',
      set(symbols['country']) - set(country['name']))

Unmappable countries after the fix: set()


### Create the dataframes required by the star schema

With clean data we can build the four dimension tables with their surrogate keys and the fact table, valuing the foreign keys through lookups. For `Dim_Time` we use day granularity; the source records timestamps down to the second, but the hierarchy starts at the Day level and no analysis needs the time of day, so we drop it during transformation.

In [6]:
# ============================================================
# 12. Dim_Time (grain: one row per calendar day)
# ============================================================

# Parse the timestamps (dd/mm/yyyy hh:mm:ss) into proper datetimes.
ts = pd.to_datetime(transactions['Date'], format='%d/%m/%Y %H:%M:%S')

# Keep only the calendar day: this is the granularity of Dim_Time.
transactions['FullDate'] = ts.dt.normalize()

# One row per distinct trading day found in the data.
dim_time = pd.DataFrame({'Date': sorted(transactions['FullDate'].unique())})

# Derive the hierarchy levels and the DayOfWeek branch attribute.
dim_time['DayOfWeek'] = dim_time['Date'].dt.day_name()
dim_time['Month']     = dim_time['Date'].dt.month
dim_time['Quarter']   = dim_time['Date'].dt.quarter
dim_time['Year']      = dim_time['Date'].dt.year

# Surrogate key: progressive integer
dim_time.insert(0, 'Time_SK', range(1, len(dim_time) + 1))

print(dim_time.shape)
dim_time.head()

(232, 6)


,Time_SK,Date,DayOfWeek,Month,Quarter,Year
0,1,2024-01-02,Tuesday,1.0,1.0,2024.0
1,2,2024-01-03,Wednesday,1.0,1.0,2024.0
2,3,2024-01-04,Thursday,1.0,1.0,2024.0
3,4,2024-01-05,Friday,1.0,1.0,2024.0
4,5,2024-01-08,Monday,1.0,1.0,2024.0


In [13]:
# ============================================================
# 13. Dim_Geography (Country -> Sub-region -> Region)
# ============================================================

# Start from the distinct countries of the symbols master data,
# join them with country.csv to bring in the two hierarchy levels,
# and rename the columns to the names of the star schema.
dim_geography = (symbols[['country']].drop_duplicates()
                 .merge(country, left_on='country', right_on='name')
                 [['country', 'sub-region', 'region']]
                 .rename(columns={'country': 'Country',
                                  'sub-region': 'SubRegion',
                                  'region': 'Region'})
                 .sort_values('Country').reset_index(drop=True))

# Surrogate key.
dim_geography.insert(0, 'Geography_SK', range(1, len(dim_geography) + 1))

print(dim_geography.shape)
dim_geography.head()

(42, 4)


,Geography_SK,Country,SubRegion,Region
0,1,Australia,Australia and New Zealand,Oceania
1,2,Bahamas,Latin America and the Caribbean,Americas
2,3,Bermuda,Northern America,Americas
3,4,Brazil,Latin America and the Caribbean,Americas
4,5,Canada,Northern America,Americas


In [14]:
# ============================================================
# 14. Dim_Symbol and Dim_TransactionType
# ============================================================

# Dim_Symbol: ticker, company name (descriptive), industry, sector.
# The country column stays OUT of this dimension: it belongs to
# Dim_Geography, to avoid duplicating the same attribute (see 1.2).
dim_symbol = (symbols[['symbol', 'company_name', 'industry', 'sector']]
              .rename(columns={'symbol': 'Symbol', 'company_name': 'CompanyName',
                               'industry': 'Industry', 'sector': 'Sector'})
              .reset_index(drop=True))
dim_symbol.insert(0, 'Symbol_SK', range(1, len(dim_symbol) + 1))

# Dim_TransactionType: one row per distinct type (BUY / SELL / DIVIDEND).
dim_transactiontype = pd.DataFrame(
    {'TransactionType': sorted(transactions['TransactionType'].unique())})
dim_transactiontype.insert(0, 'TransactionType_SK',
                           range(1, len(dim_transactiontype) + 1))

print(dim_symbol.shape)
dim_transactiontype

(3194, 5)


,TransactionType_SK,TransactionType
0,1,BUY
1,2,DIVIDEND
2,3,SELL


In [15]:
# ============================================================
# 15. Fact_Transactions: value the foreign keys via lookups
# ============================================================

# Bridge symbol -> country, needed to value the geographic FK:
# the fact reaches the geography THROUGH the traded symbol.
symbol_country = symbols[['symbol', 'country']]

# Chain of lookups: each merge replaces a source value with the
# surrogate key of the corresponding dimension.
fact = (transactions[['IDTransaction', 'FullDate', 'TransactionType', 'Symbol', 'Unit']]
        .merge(dim_time[['Time_SK', 'Date']], left_on='FullDate', right_on='Date')
        .merge(dim_symbol[['Symbol_SK', 'Symbol']], on='Symbol')
        .merge(symbol_country, left_on='Symbol', right_on='symbol')
        .merge(dim_geography[['Geography_SK', 'Country']],
               left_on='country', right_on='Country')
        .merge(dim_transactiontype, on='TransactionType'))

# Keep only the star schema columns: 4 FKs + descriptive attribute + measure.
fact_transactions = fact[['Time_SK', 'Geography_SK', 'Symbol_SK',
                          'TransactionType_SK', 'IDTransaction', 'Unit']].copy()

# Surrogate primary key of the fact table (see the remark in 1.4).
fact_transactions.insert(0, 'Transaction_SK', range(1, len(fact_transactions) + 1))

print(fact_transactions.shape)
fact_transactions.head()

(2069, 7)


,Transaction_SK,Time_SK,Geography_SK,Symbol_SK,TransactionType_SK,IDTransaction,Unit
0,1,8,32,284,1,2769834124,1605
1,2,17,32,284,3,2767324642,1605
2,3,7,32,284,3,2815473914,914
3,4,11,3,4,1,2622244212,646
4,5,11,26,258,3,2629871124,646


In [16]:
# ============================================================
# 16. Final integrity checks on the star schema
# ============================================================

# One fact row per valid transaction.
assert len(fact_transactions) == len(transactions)

# No null foreign key: referential integrity towards the dimensions.
assert fact_transactions[['Time_SK', 'Geography_SK', 'Symbol_SK',
                          'TransactionType_SK']].notna().all().all()

# The surrogate key of the fact table is unique.
assert fact_transactions['Transaction_SK'].is_unique

print('Fact_Transactions   :', len(fact_transactions), 'rows')
print('Dim_Time            :', len(dim_time), 'rows')
print('Dim_Geography       :', len(dim_geography), 'rows')
print('Dim_Symbol          :', len(dim_symbol), 'rows')
print('Dim_TransactionType :', len(dim_transactiontype), 'rows')
print('Null FKs            : 0 -> referential integrity OK')

Fact_Transactions   : 2069 rows
Dim_Time            : 228 rows
Dim_Geography       : 42 rows
Dim_Symbol          : 3194 rows
Dim_TransactionType : 3 rows
Null FKs            : 0 -> referential integrity OK


## 2.2 Analytical Questions

I selected the 8 questions with one criterion: avoid repeating the same kind of query over and over. Together, the eight questions use all four dimensions of the schema, touch different hierarchy levels (symbol, industry and sector; country, sub-region and region; quarter and year) and measure in different ways. Some count transactions (COUNT), others sum the traded units (SUM), one computes an average (AVG). This way every part of the model built in Part 1 is actually put to the test.

| # | Question | Levels used | Aggregation |
|---|---|---|---|
| 1 | Top 5 sectors by number of SELL transactions in the US | Sector, Country, Year | COUNT |
| 2 | Rank the quarters by number of transactions | Quarter | COUNT |
| 3 | Top 10 countries by number of SELL transactions | Country | COUNT |
| 4 | Top 5 regions by units bought | Region | SUM |
| 5 | Top 5 industries by units traded in Asia | Industry, Region | SUM |
| 6 | Top 5 companies by average units per transaction in Q2 | Company (Symbol), Quarter | AVG |
| 7 | Top 5 sub-regions by number of transactions in Q4 | Sub-region, Quarter | COUNT |
| 8 | Top 3 sectors by units sold on Mondays | Sector, DayOfWeek | SUM |

One preliminary step is needed before answering. The questions combine attributes that live in different tables (the sector is in `Dim_Symbol`, the quarter in `Dim_Time`, the region in `Dim_Geography`) so the fact table must be joined with its dimensions. On this single table, each question comes down to a filter and a `groupby`.

In [17]:
# ============================================================
# 17. Build the multidimensional view (fact JOIN dimensions)
# ============================================================

# One join per dimension is enough to reconstruct the whole
# multidimensional view (this is the strength of the star schema).
star = (fact_transactions
        .merge(dim_time, on='Time_SK')
        .merge(dim_geography, on='Geography_SK')
        .merge(dim_symbol, on='Symbol_SK')
        .merge(dim_transactiontype, on='TransactionType_SK'))

print(star.shape)
star.head(3)

(2069, 20)


,Transaction_SK,Time_SK,Geography_SK,Symbol_SK,TransactionType_SK,IDTransaction,Unit,Date,DayOfWeek,Month,Quarter,Year,Country,SubRegion,Region,Symbol,CompanyName,Industry,Sector,TransactionType
0,1,8,32,284,1,2769834124,1605,2024-01-11,Thursday,1,1,2024,Peru,Latin America and the Caribbean,Americas,BAP,Credicorp Ltd.,Banks - Regional,Financial Services,BUY
1,2,17,32,284,3,2767324642,1605,2024-01-24,Wednesday,1,1,2024,Peru,Latin America and the Caribbean,Americas,BAP,Credicorp Ltd.,Banks - Regional,Financial Services,SELL
2,3,7,32,284,3,2815473914,914,2024-01-10,Wednesday,1,1,2024,Peru,Latin America and the Caribbean,Americas,BAP,Credicorp Ltd.,Banks - Regional,Financial Services,SELL


# Analytical Question 1

## Top 5 sectors by number of SELL transactions in the US during 2024

> **What are the top 5 sectors by number of SELL transactions in US during 2024?**

- **Dimensions**: Symbol (Sector level), Geography (Country level), TransactionType, Time (Year level)
- **Measure**: number of transactions (COUNT of the primary events)

In [18]:
# ============================================================
# 18. Q1: top 5 sectors by SELL count in the US
# ============================================================

# Slice: SELL transactions only, US companies only, year 2024.
# Roll-up: group by sector and COUNT the events.
q1 = (star[(star['TransactionType'] == 'SELL') &
           (star['Country'] == 'United States of America') &
           (star['Year'] == 2024)]
      .groupby('Sector').size()                  # COUNT of primary events
      .sort_values(ascending=False).head(5)      # top 5
      .rename('SELL_transactions'))
q1

Sector
Technology                158
Communication Services     58
Financial Services         55
Healthcare                 50
Consumer Cyclical          48
Name: SELL_transactions, dtype: int64

### Interpretation

Technology clearly dominates the SELL side of the US holdings (158 transactions, almost three times the second sector), while the next four sectors move in a similar range (48–58). The portfolio is clearly tilted towards US tech names.

# Analytical Question 2

## Rank all quarters of 2024 by total number of transactions

> **Rank all quarters of 2024 by total number of transactions (BUY + SELL).**

- **Dimensions**: Time (Quarter level), TransactionType
- **Measure**: number of transactions (COUNT)

In [19]:
# ============================================================
# 19. Q2: ranking of the quarters by BUY + SELL count
# ============================================================

# Slice: BUY and SELL only, as required by the question text.
# Roll-up: group by quarter and COUNT the events.
q2 = (star[star['TransactionType'].isin(['BUY', 'SELL'])]
      .groupby('Quarter').size()
      .sort_values(ascending=False)
      .rename('transactions'))
q2

Quarter
1    968
2    522
3    242
4    241
Name: transactions, dtype: int64

### Interpretation

Trading activity is strongly concentrated at the beginning of the year: Q1 alone (968 operations) is worth almost as much as the other three quarters combined, and the volume decreases monotonically down to Q3 and Q4, which are practically tied around 240 operations.

# Analytical Question 3

## Top 10 countries by number of SELL transactions in 2024

> **What are the top 10 countries by number of SELL transactions in 2024?**

- **Dimensions**: Geography (Country level, the finest of the hierarchy), TransactionType, Time (Year)
- **Measure**: number of transactions (COUNT)

In [20]:
# ============================================================
# 20. Q3: top 10 countries by SELL count
# ============================================================

# Slice: SELL only, year 2024.
# Roll-up: group by country and COUNT the events.
q3 = (star[(star['TransactionType'] == 'SELL') & (star['Year'] == 2024)]
      .groupby('Country').size()
      .sort_values(ascending=False).head(10)
      .rename('SELL_transactions'))
q3

Country
United States of America                                389
United Kingdom of Great Britain and Northern Ireland    130
China                                                   112
Brazil                                                   69
Taiwan, Province of China                                50
Netherlands, Kingdom of the                              46
Switzerland                                              37
Ireland                                                  31
Luxembourg                                               27
Canada                                                   22
Name: SELL_transactions, dtype: int64

### Interpretation

The United States leads by a wide margin (389 sells), followed by the United Kingdom and China; the tail of the top 10 sits between 20 and 70 operations. The ranking reflects the country of the issuing company, not the listing market: this is why countries such as Taiwan and Luxembourg appear.

# Analytical Question 4

## Top 5 regions by total units bought in 2024

> **What are the top 5 regions by total units bought in 2024?**

- **Dimensions**: Geography (Region level, the most aggregated), TransactionType, Time (Year)
- **Measure**: `Unit` (SUM, additive measure)

In [21]:
# ============================================================
# 21. Q4: top 5 regions by units bought
# ============================================================

# Slice: BUY only, year 2024.
# Roll-up: group by region and SUM the units (additive measure).
q4 = (star[(star['TransactionType'] == 'BUY') & (star['Year'] == 2024)]
      .groupby('Region')['Unit'].sum()
      .sort_values(ascending=False).head(5))
q4

Region
Americas    37026
Europe      22528
Asia        11537
Name: Unit, dtype: int64

### Interpretation

Rolling up to the most aggregated level of the geographic hierarchy produces only three groups: the portfolio only contains companies from the Americas, Europe and Asia, so the "top 5" reduces to three rows. The Americas absorb more than half of the units bought. Without the fix of Taiwan's empty region done in 2.1, Asia would have silently lost more than two thousand units in this count.

# Analytical Question 5

## Top 5 industries by units traded in Asia in 2024

> **What are the top 5 industries by units traded in Asia in 2024?**

- **Dimensions**: Symbol (Industry level, intermediate), Geography (slice on Region = Asia), TransactionType, Time (Year)
- **Measure**: `Unit` (SUM over BUY + SELL)

In [22]:
# ============================================================
# 22. Q5: top 5 industries by units traded in Asia
# ============================================================

# Slice: Asian companies, BUY and SELL, year 2024.
# Roll-up: group by industry and SUM the units.
q5 = (star[(star['Region'] == 'Asia') &
           (star['TransactionType'].isin(['BUY', 'SELL'])) &
           (star['Year'] == 2024)]
      .groupby('Industry')['Unit'].sum()
      .sort_values(ascending=False).head(5))
q5

Industry
Semiconductors                    5401
Auto Manufacturers                4769
Diagnostics & Research            1945
Internet Content & Information    1805
Aerospace & Defense               1799
Name: Unit, dtype: int64

### Interpretation

In Asia, Semiconductors is the first industry by units traded (5,401), ahead of Auto Manufacturers: the Taiwanese tickers drive this result, which confirms that fixing Taiwan's region in 2.1 was necessary. Without it, the leading industry would have disappeared from this ranking.

# Analytical Question 6

## Top 5 companies with the highest average units per transaction in Q2

> **What are the top 5 companies with the highest average units per transaction in Q2?**

- **Dimensions**: Symbol (CompanyName, 1:1 with the Symbol level), Time (Quarter level)
- **Measure**: `Unit`, aggregated with AVG instead of SUM: the average per transaction shows how an additive measure can also be combined with other operators when the question requires it

In [23]:
# ============================================================
# 23. Q6: top 5 companies by average units per transaction in Q2
# ============================================================

# Slice: second quarter only.
# Roll-up: group by company and take the AVERAGE of the units.
q6 = (star[star['Quarter'] == 2]
      .groupby('CompanyName')['Unit'].mean()
      .sort_values(ascending=False).head(5)
      .round(2))
q6

CompanyName
Paysafe Limited                    248.33
HANGZHOU TIGERMED CONSULTING CO    208.50
Arch Capital Group Ltd.            206.50
Alvotech                           206.50
CNH Industrial N.V.                179.33
Name: Unit, dtype: float64

### Interpretation

The ranking by average trade size is led by Paysafe (248 units per operation) and is populated by names that rarely appear in the other rankings: a high average signals few large trades, not necessarily a heavily traded stock. This is why averages and counts tell different stories.

# Analytical Question 7

## Top 5 sub-regions by number of transactions in Q4 of 2024

> **What are the top 5 sub-regions by number of transactions in Q4 of 2024?**

- **Dimensions**: Geography (Sub-region level, the intermediate one made possible by the hierarchy corrected in 1.3), Time (Quarter level)
- **Measure**: number of transactions (COUNT, here over all transaction types)

In [24]:
# ============================================================
# 24. Q7: top 5 sub-regions by transaction count in Q4
# ============================================================

# Slice: fourth quarter.
# Roll-up: group by sub-region and COUNT the events.
q7 = (star[star['Quarter'] == 4]
      .groupby('SubRegion').size()
      .sort_values(ascending=False).head(5)
      .rename('transactions'))
q7

SubRegion
Northern America                   130
Eastern Asia                        41
Northern Europe                     37
Western Europe                      24
Latin America and the Caribbean     23
Name: transactions, dtype: int64

### Interpretation

In the fourth quarter Northern America is far ahead of every other sub-region (130 operations against 41 for Eastern Asia). This query is possible precisely because sub-region was modeled as the intermediate level between country and region: with the hierarchy as written in the assignment it would have been more aggregated than region, contradicting the data.

# Analytical Question 8

## Top 3 sectors by total units sold on Mondays across 2024

> **What are the top 3 sectors by total number of units sold on Mondays across 2024?**

- **Dimensions**: Symbol (Sector level), Time (selection through `DayOfWeek`, the branch attribute of the Day level), TransactionType
- **Measure**: `Unit` (SUM over SELL only)

In [25]:
# ============================================================
# 25. Q8: top 3 sectors by units sold on Mondays
# ============================================================

# Slice: SELL transactions, Mondays only, year 2024.
# Roll-up: group by sector and SUM the units.
q8 = (star[(star['TransactionType'] == 'SELL') &
           (star['DayOfWeek'] == 'Monday') &
           (star['Year'] == 2024)]
      .groupby('Sector')['Unit'].sum()
      .sort_values(ascending=False).head(3))
q8

Sector
Technology           4361
Healthcare           2108
Consumer Cyclical    1218
Name: Unit, dtype: int64

### Interpretation

Even restricting to Mondays the picture does not change: Technology first (4,361 units sold), then Healthcare and Consumer Cyclical. The question shows the usefulness of branch attributes: `DayOfWeek` is not used to climb the hierarchy, but it enables selections that the Day → Month → Quarter → Year path alone would not allow.

---

One last observation about the numbers. The answers are consistent with each other and with 2.1: the BUY + SELL operations of question 2 (968 + 522 + 242 + 241 = 1973), added to the 96 dividends, give exactly the 2069 rows of the fact table loaded in 2.1. The loading and the queries are therefore working on the same clean data.